# Empirical Ordinal-Grid Landscape

When the variables are numeric and sampled on a discrete grid (compositions, temperatures, pH, concentrations, …), they should be declared `ordinal`. The encoded codes then preserve value ordering.

As a demo, we use the W-Re-Os refractory alloy library from:

Yong Wang *et al.*, "High-throughput discovery of ultrahigh-temperature multi-principal element alloys by combinatorial additive manufacturing," *Nat. Commun.* **17**, 668 (2026). 496 alloys on a 3 at.% ternary simplex; fitness is the paper's TOPSIS composite score `Ci`.

The compositions obey `W + Re + Os = 100`, so we build the landscape over `(W, Re)` and let `Os` be implicit — otherwise no two configurations could ever differ in exactly one axis.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Materials/WReOs/simplex.csv')
df.head()

## 2. Build the landscape

In [ ]:
from graphfla.landscape import Landscape

X = df[['W', 'Re']]
f = df['TOPSIS_Ci']

landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={'W': 'ordinal', 'Re': 'ordinal'},
    verbose=True,
)

## 3. Basic information

In [ ]:
print(f"Size: {len(landscape)}")
print(f"Dimension: {landscape.n_vars}")
print(f"Number of local optima: {landscape.n_lo}")
print(f"Global optimum index: {landscape.go_index}")
print("Global optimum information:")
print(landscape[landscape.go_index])

## 4. Spatial fitness map

With only two axes, the landscape can be rendered directly.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(df['W'], df['Re'], c=df['TOPSIS_Ci'], cmap='viridis', s=14)
ax.set_xlabel('W (at. %)'); ax.set_ylabel('Re (at. %)')
plt.colorbar(sc, ax=ax, label='TOPSIS Ci')
plt.tight_layout()

## 5. Landscape features

In [ ]:
from graphfla.analysis import (
    local_optima_ratio, autocorrelation, fdc,
)

print(f"lo_ratio:        {local_optima_ratio(landscape):.4f}")
print(f"autocorrelation: {autocorrelation(landscape):.4f}")
print(f"fdc:             {fdc(landscape):.4f}")


### The whole feature panel in one call

`analysis.profile()` runs every whole-landscape metric and returns a tidy `pandas` Series — the computed-metric companion to `landscape.describe()`. Pass a list of landscapes to get a `DataFrame`, one row each, for side-by-side comparison.

In [ ]:
from graphfla import analysis

# one call -> a pandas Series of every whole-landscape metric
analysis.profile(landscape)